# 02: Feature embedding

Feature embedding converts the extracted tensors into three learned representations:

- `m`: information about every residue in each selected sequence
- `z`: information about every pair of query residues
- `e`: a smaller representation of the additional sequences

This stage uses linear projections and broadcasting. Attention begins in the Evoformer.

**Read alongside:** `../src/af2_from_scratch/feature_embedding.py`


## Stage map

```text
query sequence
     |
     +-----------------------> initial residue-pair representation z
     |                               (every residue i paired with j)
     |
main MSA features -----------> sequence representation m

extra MSA features ----------> smaller extra-sequence representation e

previous query representation + previous pair representation
                              |
                              v
                    next recycling pass
```

Embedding changes the feature width and creates the pair grid. It does not perform attention.


In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
from af2_from_scratch.feature_extraction import msa_features, sample_batch
from af2_from_scratch.feature_embedding import InputEmbedder
from af2_from_scratch import AF2Config

cfg = AF2Config()
f = msa_features("../examples/tautomerase/alignment.a3m")
b = sample_batch(f, cfg.n_clu, cfg.n_ext, seed=0)
emb = InputEmbedder(cfg)

## 1. Build the initial pair representation

The model applies two different learned projections to the query sequence. One represents residue `i` in the row role and the other represents residue `j` in the column role. Broadcasting combines every `i` with every `j`, producing the pair grid `z`.

```text
query residue i ---> row projection ----+
                                         +---> z[i, j]
query residue j ---> column projection -+
```

This is an **outer sum**, not an outer product. Because the row and column projections have different parameters, `z[i, j]` and `z[j, i]` may initially differ.


In [ ]:
target = b["target_feat"]
row_features = emb.tf_i(target)
column_features = emb.tf_j(target)
z_outer = row_features[:, None] + column_features[None]

print(
    "row features:",
    tuple(row_features.shape),
    "+ column features:",
    tuple(column_features.shape),
    "-> pair grid:",
    tuple(z_outer.shape),
)
plt.imshow(z_outer[..., 0].detach(), cmap="RdBu", vmin=-1, vmax=1)
plt.title("z[:, :, 0]: row feature i + column feature j")
plt.colorbar()
plt.show()

## 2. Add relative sequence positions

The pair representation also needs to know where residues occur in the original sequence. For every pair, `relpos` calculates the signed offset `i - j`, limits it to `[-32, 32]`, converts it into a category, and learns a vector for that category.

```text
residue indices
      |
      v
signed offset i - j
      |
      v
clip to [-32, 32]
      |
      v
learned relative-position vector
      |
      v
add to z[i, j]
```

The signed offset preserves whether residue `i` comes before or after residue `j`. Offsets beyond 32 positions share the same boundary category.


In [ ]:
relative_position_features = emb.relpos(b["residue_index"])
plt.imshow(relative_position_features[..., 0].detach(), cmap="viridis")
plt.title("Relative-position channel 0: signed sequence offset i - j")
plt.colorbar()
plt.show()

## 3. Create all three representations

The main MSA features are projected into `m`. A query-sequence embedding is added to every MSA row so each related sequence is interpreted relative to the query.

Extra MSA features are projected into the narrower representation `e`, making it cheaper to process many additional sequences.

With the default configuration:

```text
m: (S, R, 64)
z: (R, R, 64)
e: (E, R, 32)
```


In [ ]:
m, z, e = emb(b)
print("m:", tuple(m.shape), "  z:", tuple(z.shape), "  e:", tuple(e.shape))
print(f"embedder params: {sum(p.numel() for p in emb.parameters()) / 1e3:.0f}k")

## 4. Recycle information from the previous pass

AlphaFold runs the Evoformer and Structure Module more than once. The previous query-row representation is normalized and added to `m[0]`, while the previous pair representation is normalized and added to `z`. The first pass has no previous outputs.

```text
previous query row ---> normalize ---> add to next m[0]
previous pair grid  ---> normalize ---> add to next z
```

`model.py` detaches the previous outputs so gradients do not flow between recycling passes. Full AlphaFold also recycles predicted pseudo-beta distances into `z`; this educational implementation omits that path.


In [ ]:
from af2_from_scratch.feature_embedding import RecyclingEmbedder

recycling_embedder = RecyclingEmbedder(cfg)
recycled_m, recycled_z = recycling_embedder(
    m,
    z,
    m[0].detach(),
    z.detach(),
)

print("recycled shapes:", tuple(recycled_m.shape), tuple(recycled_z.shape))
print("query row changed:", not torch.allclose(recycled_m[0], m[0]))
print("other rows unchanged:", torch.allclose(recycled_m[1:], m[1:]))

**Recap:** Feature embedding creates the model's working representations: sequence information in `m`, residue-pair information in `z`, and additional evolutionary information in `e`.

Next, `03_evoformer.ipynb` shows how the Evoformer repeatedly exchanges information between `m` and `z`.
